# Telco Customer Analytics
**Classification • Regression • K-Means + Hierarchical Clustering**

## Run Instructions
1. Install dependencies: `pip install -r ../requirements.txt`
2. Configure Kaggle credentials:
   - `~/.kaggle/kaggle.json`, or
   - `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables.
3. Run this notebook top-to-bottom.


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
    silhouette_samples,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA

from scipy.cluster.hierarchy import dendrogram, linkage
import statsmodels.api as sm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load .env if present (KAGGLE_USERNAME/KAGGLE_KEY)
def load_env_from_candidates():
    candidates = []
    cwd = Path.cwd()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    if cwd.name == "notebooks":
        candidates.append(cwd.parent)

    if (cwd / "notebooks").exists():
        candidates.append(cwd / "notebooks")
        candidates.extend((cwd / "notebooks").parents)

    env_root = os.environ.get("TELCO_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))

    # Fallback to known project path (adjust if moved)
    candidates.append(Path.home() / "Desktop/EDU/1.Course work/3.term 3/analytics/proj")

    for p in candidates:
        env_path = p / ".env"
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if line.startswith("export "):
                    line = line[len("export "):]
                if "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                os.environ.setdefault(key, value)
            return env_path
    return None


def ensure_kaggle_json():
    user = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")
    if not user or not key:
        return None
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    if not kaggle_json.exists():
        kaggle_json.write_text('{"username": "%s", "key": "%s"}' % (user, key))
        try:
            kaggle_json.chmod(0o600)
        except Exception:
            pass
    return kaggle_json

# Try loading .env on import
load_env_from_candidates()

# Load .env if present (KAGGLE_USERNAME/KAGGLE_KEY)
def load_env(start_path: Path):
    for p in [start_path] + list(start_path.parents):
        env_path = p / ".env"
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if line.startswith("export "):
                    line = line[len("export "):]
                if "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                os.environ.setdefault(key, value)
            return env_path
    return None

_ = load_env(PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

DATASET_SLUG = "blastchar/telco-customer-churn"  # Change if needed
DATA_FILENAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")


## 1) Download Dataset from Kaggle


In [ ]:
# Ensure kaggle package is available
try:
    import kaggle  # noqa: F401
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])


In [ ]:
# Ensure .env is loaded even if setup cell wasn't run
try:
    load_env_from_candidates  # noqa: F401
except NameError:
    def load_env_from_candidates():
        candidates = []
        cwd = Path.cwd()
        candidates.append(cwd)
        candidates.extend(cwd.parents)
        if cwd.name == "notebooks":
            candidates.append(cwd.parent)
        if (cwd / "notebooks").exists():
            candidates.append(cwd / "notebooks")
            candidates.extend((cwd / "notebooks").parents)
        env_root = os.environ.get("TELCO_PROJECT_ROOT")
        if env_root:
            candidates.append(Path(env_root))
        candidates.append(Path.home() / "Desktop/EDU/1.Course work/3.term 3/analytics/proj")
        for p in candidates:
            env_path = p / ".env"
            if env_path.exists():
                for line in env_path.read_text().splitlines():
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    if line.startswith("export "):
                        line = line[len("export "):]
                    if "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    os.environ.setdefault(key, value)
                return env_path
        return None


def ensure_kaggle_json():
    user = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")
    if not user or not key:
        return None
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    if not kaggle_json.exists():
        kaggle_json.write_text('{"username": "%s", "key": "%s"}' % (user, key))
        try:
            kaggle_json.chmod(0o600)
        except Exception:
            pass
    return kaggle_json

load_env_from_candidates()
ensure_kaggle_json()

# Ensure .env is loaded even if setup cell wasn't run
try:
    load_env  # noqa: F401
except NameError:
    def load_env(start_path: Path):
        for p in [start_path] + list(start_path.parents):
            env_path = p / ".env"
            if env_path.exists():
                for line in env_path.read_text().splitlines():
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    if line.startswith("export "):
                        line = line[len("export "):]
                    if "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    os.environ.setdefault(key, value)
                return env_path
        return None

load_env(Path.cwd())

from kaggle.api.kaggle_api_extended import KaggleApi


def kaggle_credentials_present():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    return kaggle_json.exists() or ("KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ)

if not kaggle_credentials_present():
    raise RuntimeError(
        "Kaggle credentials not found. Place kaggle.json in ~/.kaggle/ or set KAGGLE_USERNAME and KAGGLE_KEY."
    )

api = KaggleApi()
api.authenticate()

csv_path = DATA_RAW / DATA_FILENAME
if not csv_path.exists():
    print(f"Downloading dataset: {DATASET_SLUG} ...")
    api.dataset_download_files(DATASET_SLUG, path=DATA_RAW, unzip=True)
    # Attempt to locate CSV by name or pattern
    if not csv_path.exists():
        candidates = list(DATA_RAW.glob("*.csv"))
        if candidates:
            csv_path = candidates[0]

if not csv_path.exists():
    raise FileNotFoundError("CSV not found after download. Check DATASET_SLUG and DATA_FILENAME.")

print(f"Using dataset: {csv_path}")


## 2) Load & Inspect


In [ ]:
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
display(df.head())
df.info()
display(df.isna().sum().sort_values(ascending=False))


## 3) Cleaning & Feature Engineering


In [ ]:
# Convert TotalCharges to numeric, coerce errors to NaN
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Convert SeniorCitizen to Yes/No
if "SeniorCitizen" in df.columns:
    df["SeniorCitizen"] = df["SeniorCitizen"].map({1: "Yes", 0: "No"})

# Drop customerID if present
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

# Create num_services feature
service_flags = pd.DataFrame({
    "PhoneService": df["PhoneService"].eq("Yes"),
    "MultipleLines": df["MultipleLines"].eq("Yes"),
    "InternetService": df["InternetService"].ne("No"),
    "OnlineSecurity": df["OnlineSecurity"].eq("Yes"),
    "OnlineBackup": df["OnlineBackup"].eq("Yes"),
    "DeviceProtection": df["DeviceProtection"].eq("Yes"),
    "TechSupport": df["TechSupport"].eq("Yes"),
    "StreamingTV": df["StreamingTV"].eq("Yes"),
    "StreamingMovies": df["StreamingMovies"].eq("Yes"),
})

df["num_services"] = service_flags.sum(axis=1)

# Tenure groups for EDA
if "tenure" in df.columns:
    df["tenure_group"] = pd.cut(
        df["tenure"],
        bins=[0, 12, 24, 36, 48, 60, 72],
        labels=["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"],
        include_lowest=True,
    )

print("Cleaned shape:", df.shape)
display(df.head())


## 4) Exploratory Data Analysis (EDA)


In [ ]:
# Churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", palette="Set2")
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate by Contract and PaymentMethod
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contract_rate = df.groupby("Contract")["Churn"].apply(lambda x: (x == "Yes").mean()).reset_index()
contract_rate["ChurnRate"] = contract_rate["Churn"].astype(float)

sns.barplot(data=contract_rate, x="Contract", y="ChurnRate", ax=axes[0], palette="viridis")
axes[0].set_title("Churn Rate by Contract")
axes[0].set_ylabel("Churn Rate")
axes[0].set_xlabel("Contract")
axes[0].tick_params(axis='x', rotation=15)

payment_rate = df.groupby("PaymentMethod")["Churn"].apply(lambda x: (x == "Yes").mean()).reset_index()
payment_rate["ChurnRate"] = payment_rate["Churn"].astype(float)

sns.barplot(data=payment_rate, x="PaymentMethod", y="ChurnRate", ax=axes[1], palette="magma")
axes[1].set_title("Churn Rate by Payment Method")
axes[1].set_ylabel("Churn Rate")
axes[1].set_xlabel("Payment Method")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Numeric distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["tenure"], kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Tenure Distribution")

sns.histplot(df["MonthlyCharges"], kde=True, ax=axes[1], color="#55A868")
axes[1].set_title("Monthly Charges Distribution")

sns.histplot(df["TotalCharges"], kde=True, ax=axes[2], color="#C44E52")
axes[2].set_title("Total Charges Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots by Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[0], palette="Set3")
axes[0].set_title("Monthly Charges by Churn")

sns.boxplot(data=df, x="Churn", y="TotalCharges", ax=axes[1], palette="Set3")
axes[1].set_title("Total Charges by Churn")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap for numeric features
numeric_df = df.select_dtypes(include=["number"]).copy()

# Include churn as numeric for correlation
if "Churn" in df.columns:
    numeric_df["Churn"] = (df["Churn"] == "Yes").astype(int)

plt.figure(figsize=(8, 6))
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()


In [ ]:
# Pairplot for key numeric features (sampled for speed)
sample_df = df.sample(n=min(1000, len(df)), random_state=RANDOM_STATE)
plot_cols = ["tenure", "MonthlyCharges", "TotalCharges", "num_services", "Churn"]
sns.pairplot(sample_df[plot_cols], hue="Churn", corner=True, plot_kws={"alpha": 0.6})
plt.show()


In [ ]:
# num_services vs Churn
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Churn", y="num_services", palette="Set2")
plt.title("Number of Services by Churn")
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate by tenure group (visual)
if "tenure_group" in df.columns:
    churn_by_tenure = df.groupby("tenure_group")["Churn"].apply(lambda x: (x == "Yes").mean()).reset_index()
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=churn_by_tenure, x="tenure_group", y="Churn", marker="o")
    plt.title("Churn Rate by Tenure Group")
    plt.xlabel("Tenure Group (months)")
    plt.ylabel("Churn Rate")
    plt.tight_layout()
    plt.show()


## 5) Preprocessing Helpers


In [ ]:
# Drop EDA-only column for modeling
model_df = df.drop(columns=["tenure_group"], errors="ignore")


def build_preprocessor(X):
    categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
    numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numeric_features),
        ]
    )

    return preprocessor, categorical_features, numeric_features


## 6) Classification: Churn Prediction


In [ ]:
# Prepare features and target
y = (model_df["Churn"] == "Yes").astype(int)
X = model_df.drop(columns=["Churn"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

preprocessor, cat_features, num_features = build_preprocessor(X_train)


def make_pipeline(model):
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])

models = {
    "LogisticRegression": make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "DecisionTree": make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    "RandomForest": make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced_subsample")),
    "HistGradientBoosting": make_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
}

param_grids = {
    "DecisionTree": {
        "model__max_depth": [3, 5, 10, None],
        "model__min_samples_split": [2, 5, 10],
    },
    "RandomForest": {
        "model__n_estimators": [200],
        "model__max_depth": [None, 10],
        "model__min_samples_split": [2, 5],
    },
}

results = []
fitted = {}

for name, pipeline in models.items():
    if name in param_grids:
        grid = GridSearchCV(
            pipeline,
            param_grids[name],
            cv=5,
            scoring="f1",
            n_jobs=-1,
        )
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
    else:
        pipeline.fit(X_train, y_train)
        best_model = pipeline

    fitted[name] = best_model
    y_pred = best_model.predict(X_test)
    if hasattr(best_model, "predict_proba"):
        y_prob = best_model.predict_proba(X_test)[:, 1]
    else:
        y_prob = best_model.decision_function(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
display(results_df)


In [ ]:
# Pick best model by F1
best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
print("Best model:", best_name)

# Confusion matrix
cm = confusion_matrix(y_test, best_model.predict(X_test))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# ROC curve
if hasattr(best_model, "predict_proba"):
    y_prob = best_model.predict_proba(X_test)[:, 1]
else:
    y_prob = best_model.decision_function(X_test)

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title(f"ROC Curve - {best_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
plt.figure(figsize=(6, 4))
plt.plot(rec, prec, label=f"AP = {avg_prec:.3f}")
plt.title(f"Precision-Recall Curve - {best_name}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance (for tree-based) or coefficients (for logistic)
preprocess = best_model.named_steps["preprocess"]

# Extract feature names
cat_feature_names = preprocess.named_transformers_["cat"].get_feature_names_out(cat_features)
feature_names = np.concatenate([cat_feature_names, num_features])

model = best_model.named_steps["model"]

if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-10:]
    top_features = feature_names[top_idx]
    top_values = importances[top_idx]

    plt.figure(figsize=(7, 4))
    sns.barplot(x=top_values, y=top_features, palette="viridis")
    plt.title(f"Top 10 Feature Importances - {best_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
elif hasattr(model, "coef_"):
    coefs = model.coef_.ravel()
    top_idx = np.argsort(np.abs(coefs))[-10:]
    top_features = feature_names[top_idx]
    top_values = coefs[top_idx]

    plt.figure(figsize=(7, 4))
    sns.barplot(x=top_values, y=top_features, palette="coolwarm")
    plt.title(f"Top 10 Coefficients - {best_name}")
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


## 7) Regression: Predict TotalCharges


In [ ]:
# Regression target and features
y_reg = model_df["TotalCharges"]
X_reg = model_df.drop(columns=["TotalCharges", "Churn"])

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_STATE
)

preprocessor_reg, cat_features_reg, num_features_reg = build_preprocessor(Xr_train)

reg_models = {
    "LinearRegression": Pipeline(steps=[("preprocess", preprocessor_reg), ("model", LinearRegression())]),
    "RandomForestRegressor": Pipeline(
        steps=[("preprocess", preprocessor_reg), ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE))]
    ),
    "GradientBoostingRegressor": Pipeline(
        steps=[("preprocess", preprocessor_reg), ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))]
    ),
}

reg_results = []
reg_fitted = {}

for name, pipeline in reg_models.items():
    pipeline.fit(Xr_train, yr_train)
    reg_fitted[name] = pipeline
    preds = pipeline.predict(Xr_test)

    reg_results.append({
        "model": name,
        "MAE": mean_absolute_error(yr_test, preds),
        "RMSE": np.sqrt(mean_squared_error(yr_test, preds)),
        "R2": r2_score(yr_test, preds),
    })

reg_results_df = pd.DataFrame(reg_results).sort_values(by="RMSE")
display(reg_results_df)


In [ ]:
# Best regression model by RMSE
best_reg_name = reg_results_df.iloc[0]["model"]
best_reg_model = reg_fitted[best_reg_name]
print("Best regression model:", best_reg_name)

preds = best_reg_model.predict(Xr_test)

# Actual vs Predicted
plt.figure(figsize=(6, 5))
plt.scatter(yr_test, preds, alpha=0.5)
plt.plot([yr_test.min(), yr_test.max()], [yr_test.min(), yr_test.max()], color="red", linestyle="--")
plt.title(f"Actual vs Predicted - {best_reg_name}")
plt.xlabel("Actual TotalCharges")
plt.ylabel("Predicted TotalCharges")
plt.tight_layout()
plt.show()

# Residuals vs Fitted
residuals = yr_test - preds
plt.figure(figsize=(6, 4))
plt.scatter(preds, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals vs Fitted")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

# Residual distribution
plt.figure(figsize=(6, 4))
sns.histplot(residuals, kde=True, color="#4C72B0")
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.tight_layout()
plt.show()

# Q-Q plot for residuals
plt.figure(figsize=(5, 5))
sm.qqplot(residuals, line="45", fit=True)
plt.title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()


## 8) Clustering: K-Means and Hierarchical


In [ ]:
# Prepare data for clustering
X_cluster = model_df.drop(columns=["Churn"])
preprocessor_cluster, cat_features_cl, num_features_cl = build_preprocessor(X_cluster)

Xc = preprocessor_cluster.fit_transform(X_cluster)

# Convert to dense if needed (for hierarchical clustering)
if sp.issparse(Xc):
    Xc_dense = Xc.toarray()
else:
    Xc_dense = Xc

print("Clustering feature matrix shape:", Xc_dense.shape)


In [ ]:
# K-Means: Evaluate K with Elbow and Silhouette
k_values = range(2, 9)
inertias = []
silhouettes = []

# Sample for silhouette score to reduce compute if needed
n_samples = Xc_dense.shape[0]
sample_size = min(2000, n_samples)
np.random.seed(RANDOM_STATE)
idx = np.random.choice(n_samples, size=sample_size, replace=False)
X_sample = Xc_dense[idx]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(Xc_dense)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sample, labels[idx])
    silhouettes.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method (Inertia)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_values), silhouettes, marker="o", color="#C44E52")
axes[1].set_title("Silhouette Score by K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print("Selected K (max silhouette):", best_k)


In [ ]:
# Fit final K-Means
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(Xc_dense)

# PCA for 2D visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(Xc_dense)

plt.figure(figsize=(6, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels, palette="tab10", s=35, alpha=0.8)
plt.title("K-Means Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# K-Means silhouette plot
sample_labels = kmeans_labels[idx]
sil_values = silhouette_samples(X_sample, sample_labels)

plt.figure(figsize=(6, 4))
plt.hist(sil_values, bins=20, color="#4C72B0")
plt.title("K-Means Silhouette Score Distribution (Sample)")
plt.xlabel("Silhouette Score")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# K-Means cluster profile (numeric means)
cluster_df = model_df.copy()
cluster_df["kmeans_cluster"] = kmeans_labels

key_numeric = ["tenure", "MonthlyCharges", "TotalCharges", "num_services"]
profile_k = cluster_df.groupby("kmeans_cluster")[key_numeric].mean().reset_index()

profile_k_melt = profile_k.melt(id_vars="kmeans_cluster", var_name="feature", value_name="value")
plt.figure(figsize=(8, 4))
sns.barplot(data=profile_k_melt, x="kmeans_cluster", y="value", hue="feature")
plt.title("K-Means Cluster Profiles (Numeric Means)")
plt.xlabel("Cluster")
plt.ylabel("Mean")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Categorical proportions for selected features
cat_features_to_plot = ["Contract", "InternetService", "PaymentMethod"]
for col in cat_features_to_plot:
    prop = (
        cluster_df.groupby("kmeans_cluster")[col]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
    )
    plt.figure(figsize=(8, 4))
    sns.barplot(data=prop, x="kmeans_cluster", y="proportion", hue=col)
    plt.title(f"K-Means Cluster Proportions by {col}")
    plt.xlabel("Cluster")
    plt.ylabel("Proportion")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Hierarchical clustering: Dendrogram (sample for speed)
sample_size = min(2000, Xc_dense.shape[0])
np.random.seed(RANDOM_STATE)
idx_h = np.random.choice(Xc_dense.shape[0], size=sample_size, replace=False)
X_sample_h = Xc_dense[idx_h]

Z = linkage(X_sample_h, method="ward")

plt.figure(figsize=(10, 5))
dendrogram(Z, truncate_mode="level", p=5)

if best_k > 1:
    cut_height = Z[-(best_k - 1), 2]
    plt.axhline(y=cut_height, color="red", linestyle="--", label=f"Cut for k={best_k}")
    plt.legend()

plt.title("Hierarchical Clustering Dendrogram (Sample)")
plt.xlabel("Sample Index or (Cluster Size)")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()


In [ ]:
# Fit Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hier_labels = agg.fit_predict(Xc_dense)

plt.figure(figsize=(6, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=hier_labels, palette="tab10", s=35, alpha=0.8)
plt.title("Hierarchical Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Hierarchical cluster profiles
cluster_df["hier_cluster"] = hier_labels

profile_h = cluster_df.groupby("hier_cluster")[key_numeric].mean().reset_index()
profile_h_melt = profile_h.melt(id_vars="hier_cluster", var_name="feature", value_name="value")

plt.figure(figsize=(8, 4))
sns.barplot(data=profile_h_melt, x="hier_cluster", y="value", hue="feature")
plt.title("Hierarchical Cluster Profiles (Numeric Means)")
plt.xlabel("Cluster")
plt.ylabel("Mean")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

for col in cat_features_to_plot:
    prop = (
        cluster_df.groupby("hier_cluster")[col]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
    )
    plt.figure(figsize=(8, 4))
    sns.barplot(data=prop, x="hier_cluster", y="proportion", hue=col)
    plt.title(f"Hierarchical Cluster Proportions by {col}")
    plt.xlabel("Cluster")
    plt.ylabel("Proportion")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
# Side-by-side comparison table (cluster labels are not aligned between methods)
summary_k = profile_k.copy()
summary_k["method"] = "kmeans"
summary_k = summary_k.set_index(["method", "kmeans_cluster"])
summary_k.index = summary_k.index.set_names(["method", "cluster"])

summary_h = profile_h.copy()
summary_h["method"] = "hierarchical"
summary_h = summary_h.set_index(["method", "hier_cluster"])
summary_h.index = summary_h.index.set_names(["method", "cluster"])

summary_all = pd.concat([summary_k, summary_h])
display(summary_all)


## 9) Conclusion
Summarize key findings for churn prediction, revenue drivers, and customer segments.
Highlight business implications and potential next steps.
